<a href="https://colab.research.google.com/github/jewon6273-creator/quant-/blob/main/dart_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DART 공시 원문 추출 검증

프로젝트 진행 가능 여부를 판단하는 테스트입니다. 순서대로 실행하세요.

**확인할 것**
1. API 키가 동작하는가
2. 공시 원문이 한글 깨짐 없이 추출되는가
3. 목차 기준으로 필요한 섹션만 잘라낼 수 있는가
4. **업종·규모가 다른 여러 기업에서 동일하게 되는가** ← 가장 중요

4번이 이 프로젝트의 최대 리스크입니다. 한 기업만 되는 건 의미가 없습니다.

## 0. 준비

API 키는 https://opendart.fss.or.kr 에서 무료로 즉시 발급됩니다.  
회원가입 → 인증키 신청 → 이메일로 40자리 키 수신.

In [ ]:
import requests, zipfile, io, re, time
from bs4 import BeautifulSoup

KEY = "8cb12c7f97011dc5cd0cb6104fab3a9d18b90f01"

assert len(KEY) == 40, f"키 길이가 이상합니다 (현재 {len(KEY)}자). 40자리 키를 넣어주세요."

In [ ]:
!pip install finance-datareader

import FinanceDataReader as fdr
import pandas as pd

codes = {'105560':'KB금융', '055550':'신한지주',
         '086790':'하나금융', '316140':'우리금융'}

dfs = []
for code, name in codes.items():
    df = fdr.DataReader(code, '2020-01-01', '2025-09-01')
    df['code'] = code
    df['name'] = name
    dfs.append(df.reset_index())

prices = pd.concat(dfs)
prices.to_csv('prices.csv', index=False)
prices.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.9 MB/s eta 0:00:00


,Date,Open,High,Low,Close,Volume,Change,code,name
0,2020-01-02,47250,47450,46450,46550,1033855,-0.023085,105560,KB금융
1,2020-01-03,47050,47800,46900,47150,1123490,0.012889,105560,KB금융
2,2020-01-06,47100,47550,46300,46600,698224,-0.011665,105560,KB금융
3,2020-01-07,46600,47350,46600,47000,817477,0.008584,105560,KB금융
4,2020-01-08,46100,46550,45800,46150,1197780,-0.018085,105560,KB금융


## 1. 기업 고유번호 사전 받기

DART는 회사명이 아니라 8자리 고유번호로 조회합니다.  
전체 목록을 한 번 받아두면 회사명으로 검색할 수 있습니다. (약 10MB, 30초)

In [ ]:
r = requests.get("https://opendart.fss.or.kr/api/corpCode.xml",
                 params={"crtfc_key": KEY}, timeout=60)

if r.headers.get("Content-Type", "").startswith("application/json"):
    raise SystemExit(f"API 오류: {r.json()}")   # 키가 잘못되면 JSON으로 에러가 옵니다

z = zipfile.ZipFile(io.BytesIO(r.content))
soup = BeautifulSoup(z.read(z.namelist()[0]).decode("utf-8"), "xml")

CORPS = {}
for item in soup.find_all("list"):
    stock = (item.stock_code.text or "").strip()
    if stock:                                    # 상장사만
        CORPS[item.corp_name.text.strip()] = item.corp_code.text.strip()

print(f"상장사 {len(CORPS)}개 로드 완료")
print("삼성전자 →", CORPS.get("삼성전자"))

상장사 3959개 로드 완료
삼성전자 → 00126380


In [ ]:
def find_corp(keyword):
    """회사명 일부로 검색"""
    return {k: v for k, v in CORPS.items() if keyword in k}

find_corp("하이닉스")

{'SK하이닉스': '00164779'}

## 2. 최근 사업보고서 찾기

In [ ]:
def latest_report(corp_code, bgn="20230101", end="20261231"):
    """정기공시(사업/반기/분기보고서) 중 가장 최근 건의 접수번호 반환"""
    r = requests.get("https://opendart.fss.or.kr/api/list.json", params={
        "crtfc_key": KEY, "corp_code": corp_code,
        "bgn_de": bgn, "end_de": end,
        "pblntf_ty": "A",          # A = 정기공시
        "page_count": 100,
    }, timeout=30).json()

    if r["status"] != "000":
        return None, f"조회 실패 ({r['status']}: {r['message']})"

    # 분기·반기 말고 '사업보고서'를 우선
    for it in r["list"]:
        if "사업보고서" in it["report_nm"]:
            return it["rcept_no"], it["report_nm"]
    it = r["list"][0]
    return it["rcept_no"], it["report_nm"]


rcept_no, name = latest_report(CORPS["삼성전자"])
print(rcept_no, name)

20260310002820 사업보고서 (2025.12)


## 3. 원문 다운로드 + 텍스트 변환

여기가 1차 관문입니다. **한글이 깨지면 여기서 걸립니다.**  
DART 원문은 인코딩이 파일마다 달라서 자동 판별을 넣었습니다.

In [ ]:
def fetch_text(rcept_no):
    """접수번호 → 본문 텍스트"""
    r = requests.get("https://opendart.fss.or.kr/api/document.xml",
                     params={"crtfc_key": KEY, "rcept_no": rcept_no}, timeout=120)

    if r.headers.get("Content-Type", "").startswith("application/json"):
        return None, f"다운로드 실패: {r.json()}"

    z = zipfile.ZipFile(io.BytesIO(r.content))
    raw_bytes = z.read(max(z.namelist(), key=lambda n: z.getinfo(n).file_size))

    # 인코딩 자동 판별: 한글 자모가 깨지지 않는 쪽을 채택
    best, best_score = None, -1
    for enc in ("utf-8", "cp949", "euc-kr"):
        try:
            s = raw_bytes.decode(enc)
        except UnicodeDecodeError:
            continue
        score = len(re.findall(r"[가-힣]", s[:200000]))
        if score > best_score:
            best, best_score = s, score

    if best is None:
        return None, "디코딩 실패"

    text = BeautifulSoup(best, "html.parser").get_text(" ")
    text = re.sub(r"\s+", " ", text).strip()
    return text, None


text, err = fetch_text(rcept_no)
print(err or f"{len(text):,}자 추출")
print("---")
print(text[:600])

/tmp/ipykernel_622/1098764774.py:26: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  text = BeautifulSoup(best, "html.parser").get_text(" ")


715,506자 추출
---
사업보고서 6.3 삼성전자주식회사 C A Y 130111-0006246 사 업 보 고 서 (제 57 기) 사업연도 2025년 01월 01일 부터 2025년 12월 31일 까지 금융위원회 한국거래소 귀중 2026년 3월 10일 제출대상법인 유형 : 주권상장법인 면제사유발생 : 해당사항 없음 회 사 명 : 삼성전자주식회사 대 표 이 사 : 전 영 현 본 점 소 재 지 : 경기도 수원시 영통구 삼성로 129(매탄동) (전 화) 031-200-1114 (홈페이지) http://www.samsung.com/sec 작 성 책 임 자 : (직 책) 재경팀장 (성 명) 김 동 욱 (전 화) 031-277-7218 목 차 【 대표이사 등의 확인 】 254q 대표이사 서명2.jpg 대표이사 확인서 I. 회사의 개요 1. 회사의 개요 가. 회사의 법적ㆍ상업적 명칭 당사의 명칭은 삼성전자주식회사이고 영문명은 Samsung Electronics Co., Ltd. 입니다. 나. 설립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식


위 출력에서 **한글 문장이 읽히면 통과**입니다.

글자수가 10만 자를 넘고 `���` 같은 깨짐이 없으면 정상입니다.

## 4. 섹션 추출

30만 자 전체가 아니라 위험 문장이 몰려 있는 구간만 잘라냅니다.  
특히 **'그 밖에 투자자 보호를 위하여 필요한 사항'** 에 소송·제재·우발부채가 모여 있습니다.

In [ ]:
SECTION_PATTERNS = {
    "사업의 내용":     r"[IVX]+\s*[.．]?\s*사업의\s*내용",
    "투자자 보호사항": r"[IVX]+\s*[.．]?\s*그\s*밖에\s*투자자\s*보호",
    "재무에 관한 사항": r"[IVX]+\s*[.．]?\s*재무에\s*관한\s*사항",
}

def find_sections(text):
    """섹션명 → 등장 위치 목록 (목차와 본문 2회 이상 등장하는 게 정상)"""
    out = {}
    for label, pat in SECTION_PATTERNS.items():
        out[label] = [m.start() for m in re.finditer(pat, text)]
    return out


hits = find_sections(text)
for label, pos in hits.items():
    print(f"{label:15s} {len(pos)}회 발견 {pos[:5]}")

사업의 내용          2회 발견 [23242, 320438]
투자자 보호사항        3회 발견 [320097, 320244, 548136]
재무에 관한 사항       5회 발견 [61112, 283404, 283480, 292434, 548756]


In [ ]:
# 마지막 등장 위치(=본문)부터 뒤로 잘라서 내용 확인
pos = hits["투자자 보호사항"]
if pos:
    print(text[pos[-1]: pos[-1] + 1500])
else:
    print("해당 섹션을 못 찾았습니다. 정규식 수정 필요.")

XI. 그 밖에 투자자 보호를 위하여 필요한 사항 1. 공시내용 진행 및 변경사항 가. 공시내용 진행 및 변경사항 (단위 : 백만 달러) 신고일자 계약내역 계약금액 변동가능성 2025.07.26 - 계약명: 반도체 위탁생산 공급계약 - 계약상대방: Tesla, Inc. - 계약기간 : 2025.7월~2033.12월 16,544 - 계약기간, 계약금액 등은 고객사 수요 및 사업 경과에 따라 변동될 수 있음- 기 공시사항에 변동사항 발생시 별도 공시 예정임 ※ 당사의 제품가격, 생산능력, 납품상황 등 영업비밀에 해당하는 정보의 누설 우려가 있는 항목(수량, 금액, 기납품액, 수주잔고)의 기재는 생략하였습니다. 2. 우발부채 등에 관한 사항 가. 중요한 소송사건등 당사는 다수의 회사 등과 정상적인 영업과정에서 발생한 소송, 분쟁 및 규제기관의 조사 등을 진행 중에 있습니다. 이에 따른 자원의 유출금액 및 시기는 불확실하며 당사의 경영진은 이러한 소송 등의 결과가 당사의 재무상태에 중요한 영향을 미치지 않을 것으로 판단하고 있습니다. 당사(삼성전자㈜, 삼성디스플레이㈜ 등)는 분할된 삼성디스플레이㈜ 등의 분할 전 채무에 관하여 연대하여 변제할 책임이 있습니다.그 밖의 우발부채 및 약정사항에 대해서는 'III. 재무에 관한 사항'의 '3. 연결재무제표주석' 및 '5. 재무제표 주석'을 참조하시기 바랍니다. 나. 채무보증내역 (단위 : 천US$, %) 법인명(채무자) 관계 채권자 내용 목적 보증시작일 보증종료일 채무보증한도 채무금액 이자율 기초 기말 기초 증감 기말 SEA 계열회사 BOA 외 지급보증 운영자금 2025-04-16 2026-12-16 1,278,000 1,278,000 - - - SEM 계열회사 BBVA 외 지급보증 운영자금 2025-06-14 2026-12-16 715,000 715,000 - - - SAMCOL 계열회사 Citibank 외 지급보증 운영자금 2025-06-14 2026-12-16 210,000 180,000 - - - SEDA 계열회사 B

## 5. 위험 키워드로 후보 문장 뽑기

실제 파이프라인의 축소판입니다. 30만 자가 몇 백 문장으로 줄어드는지 확인합니다.

In [ ]:
RISK_KEYWORDS = [
    # 감사의견
    "의견거절", "한정의견", "부적정", "감사범위", "강조사항",
    # 계속기업
    "계속기업", "존속능력", "불확실성",
    # 재무구조
    "자본잠식", "관리종목", "상장폐지", "영업손실", "결손금",
    # 소송·제재
    "소송", "제재", "과징금", "고발", "우발부채", "횡령", "배임",
    # 자금조달
    "유상증자", "전환사채", "신주인수권", "채무불이행", "연체",
    # 지배구조
    "최대주주", "주식담보", "경영권",
]

def extract_candidates(text, keywords=RISK_KEYWORDS):
    sents = re.split(r"(?<=[다요]\.)\s+", text)
    sents = [s.strip() for s in sents if 20 <= len(s.strip()) <= 400]
    return [s for s in sents if any(k in s for k in keywords)]


cands = extract_candidates(text)
print(f"전체 {len(text):,}자 → 후보 {len(cands)}문장\n")
for s in cands[:5]:
    print("·", s[:150], "\n")

전체 715,506자 → 후보 81문장

· 최대주주의 변동 2021년 4월 29일에 기존 최대주주가 소유하던 당사 주식의 상속으로 인해 최대주주가 삼성생명보험㈜으로 변동되었습니다. 

· (기준일 : 2025년 12월 31일 ) (단위 : 주, %) 변동일 최대주주명 소유주식수 지분율 변동원인 비 고 2021.04.29 삼성생명보험㈜ 1,263,050,053 21.16% 변동전 최대주주의 피상속 - ※ 소유주식수 및 지분율은 최대주주 및 그 특수관계인의 

· ☞ 최대주주 관련 자세한 사항은 'VII. 주주에 관한 사항'을 참고하시기 바랍니다. 

· 당사는 내부자금 공유를 통해 외부차입금을 최소화하여 이자율 변동으로 인한 금융비용과 불확실성을 최소화하고 있습니다. 

· 1) 제57기말 (단위 : 백만원) 구 분 3개월 이내 ~6개월 ~1년 1~5년 5년 초과 금융부채 64,213,129 648,205 1,576,875 7,866,273 4,844,847 2) 제56기말 (단위 : 백만원) 구 분 3개월 이내 ~6개월 ~1년 1~5년  



## 6. 다기업 검증 ← 이게 진짜 테스트

삼성전자 하나 되는 건 아무 의미가 없습니다.  
**업종·규모·시장이 다른 기업들에서 전부 되어야** 프로젝트가 성립합니다.

여기서 실패율이 30%를 넘으면 파서 설계를 다시 해야 합니다.

In [ ]:
def list_reports(corp_code, bgn="20230101", end="20261231"):
    """정기공시 중 사업보고서 목록을 최신순으로 반환"""
    r = requests.get("https://opendart.fss.or.kr/api/list.json", params={
        "crtfc_key": KEY, "corp_code": corp_code,
        "bgn_de": bgn, "end_de": end,
        "pblntf_ty": "A", "page_count": 100,
    }, timeout=30).json()

    if r["status"] != "000":
        return []
    return [(it["rcept_no"], it["report_nm"]) for it in r["list"]
            if "사업보고서" in it["report_nm"]]


def fetch_with_fallback(corp_code, max_try=3):
    """정정본 등으로 원문이 없으면 이전 건으로 재시도"""
    reports = list_reports(corp_code)
    if not reports:
        return None, None, "사업보고서 없음"

    last_err = None
    for rc, nm in reports[:max_try]:
        text, err = fetch_text(rc)
        if not err and text and len(text) > 50000:
            return text, nm, None
        last_err = err or f"본문 부족 ({len(text) if text else 0}자)"
        time.sleep(0.5)
    return None, None, f"{max_try}건 모두 실패 (마지막: {last_err})"


TARGETS = [
    "삼성전자", "SK하이닉스",    # 대형 제조
    "KB금융", "삼성생명",        # 금융
    "셀트리온", "HLB",           # 바이오
    "NAVER", "카카오",           # 플랫폼
    "HD현대중공업", "대한항공",   # 중후장대
]

results = []
for nm in TARGETS:
    code = CORPS.get(nm)
    if not code:
        hit = find_corp(nm)
        code = list(hit.values())[0] if hit else None
    if not code:
        results.append((nm, "✗", "회사명 못 찾음"))
        continue

    try:
        t, rname, err = fetch_with_fallback(code)
        if err:
            results.append((nm, "✗", err))
            continue

        kor = len(re.findall(r"[가-힣]", t))
        sec = sum(1 for v in find_sections(t).values() if v)
        cn  = len(extract_candidates(t))

        ok = kor > 30000 and sec >= 2 and cn > 10
        results.append((nm, "✓" if ok else "△",
                        f"{len(t):>8,}자 | 섹션 {sec}/3 | 후보 {cn:>4}문장"))
    except Exception as e:
        results.append((nm, "✗", f"{type(e).__name__}: {e}"))

    time.sleep(1)

print(f"{'기업':12s} {'':2s} {'결과'}")
print("-" * 60)
for nm, mark, msg in results:
    print(f"{nm:12s} {mark:2s} {msg}")

ok_cnt = sum(1 for _, m, _ in results if m == "✓")
print("-" * 60)
print(f"성공 {ok_cnt}/{len(results)}")
print("\n8개 이상 ✓ → 진행 가능" if ok_cnt >= 8 else "\n실패 항목 원인 확인 필요")

/tmp/ipykernel_622/1098764774.py:26: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  text = BeautifulSoup(best, "html.parser").get_text(" ")


기업              결과
------------------------------------------------------------
삼성전자         ✓   715,506자 | 섹션 3/3 | 후보   81문장
SK하이닉스       ✓   444,844자 | 섹션 3/3 | 후보   66문장
KB금융         ✗  BadZipFile: File is not a zip file
삼성생명         ✓   967,290자 | 섹션 3/3 | 후보   78문장
셀트리온         ✓   502,507자 | 섹션 3/3 | 후보   59문장
HLB          ✓   432,752자 | 섹션 3/3 | 후보  108문장
NAVER        ✓   555,810자 | 섹션 3/3 | 후보   56문장
카카오          ✓   735,324자 | 섹션 3/3 | 후보   89문장
HD현대중공업      ✓   458,434자 | 섹션 3/3 | 후보   90문장
대한항공         ✓   460,589자 | 섹션 3/3 | 후보   74문장
------------------------------------------------------------
성공 9/10

8개 이상 ✓ → 진행 가능


In [ ]:
print("fetch_with_fallback" in dir())

True


In [ ]:
code = CORPS.get("KB금융")
rc, nm = latest_report(code)
print(rc, nm)

r = requests.get("https://opendart.fss.or.kr/api/document.xml",
                 params={"crtfc_key": KEY, "rcept_no": rc})
print(r.headers.get("Content-Type"), len(r.content))
print(r.content[:300])

20260619000667 [기재정정]사업보고서 (2025.12)
application/xml;charset=UTF-8 147
b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><result><status>014</status><message>\xed\x8c\x8c\xec\x9d\xbc\xec\x9d\xb4 \xec\xa1\xb4\xec\x9e\xac\xed\x95\x98\xec\xa7\x80 \xec\x95\x8a\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.</message></result>'


## 7. 결과 저장

통과했다면 후보 문장을 CSV로 뽑아둡니다. 그대로 라벨링 시트가 됩니다.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "company": "삼성전자",
    "sentence": cands,
    "label": "",        # 팀원이 채울 칸
})
df.to_csv("candidates.csv", index=False, encoding="utf-8-sig")
print(f"{len(df)}문장 저장 완료 → candidates.csv")
df.head(10)

NameError: name 'cands' is not defined

---

## 막혔을 때

| 증상 | 원인 |
|---|---|
| `status 100` | 키가 틀림 |
| `status 020` | 일일 한도(2만건) 초과 |
| `status 013` | 해당 기간에 공시 없음 → 기간 넓히기 |
| 한글 깨짐 | 3번 셀의 인코딩 후보에 `euc-kr` 추가 확인 |
| 섹션 0회 | 목차 표기가 다른 경우. 해당 기업 원문을 직접 보고 정규식 수정 |
| 후보 문장 0개 | 문장 분리 정규식 문제. `re.split` 기준을 마침표로 완화 |